# Savings rate

There's a simple analysis available at https://www.mrmoneymustache.com/2012/01/13/the-shockingly-simple-math-behind-early-retirement/ .  This document more or less replicates that analysis, while highlighting the assumptions along the way.

## Imports

In [1]:
import math

## Problem statement

If your current expenses and income remain flat in real terms (i.e., grow commensurate with inflation), and you invest all of your savings (income - expenses) into an investment portfolio with a known return, then how long must you save until your portfolio fully covers your expenses?

### Assumptions

There are numerous assumptions needed to make this a textbook calculation, which may or may not be all that realistic.  Nevertheless, this is the starting point of the anaylsis.

- income and expenses remain constant in real terms during the savings period, and expenses remain constant during retirement
- all savings go into the investment porfolio, and are never withdrawn during the savings period
- all quantities (income, expenses, savings) are distributed at the same time monthly
- the investment portfolio provides a constant, real rate of return
- the investment portfolio supports a safe withdrawal rate at any time during the analysis, however defined

### Units

The only absolute scale in the problem is set by either the income, savings, or expenses rate.  As a matter of choice, I'm using the reference quantity as the monthly income.

## Portfolio growth with monthly compounding

Given a known investment rate of return, we can convert this to a monthly rate of compounding.  For example, if $r_y$ is the investment rate of return assuming yearly compunding, then rate of return with monthly compounding $r_m$ is found via  
$r_m = 12 \left[ \left( 1 + r_y \right)^{1 / 12} - 1 \right] \quad .$  

Given the monthly savings rate $s_m$, the value of the investment portfolio after $n \ge 1$ months of compounding, $P_n$, is given by  
$P_n = s_m \sum_{j = 1}^n \left( 1 + \frac{r_m}{12} \right)^j \quad .$  
This is a geometric series, which can be evaluated as   
##### $P_n = \frac{s_m}{\left( r_m / 12 \right)}\left( 1 + \frac{r_m}{12} \right) \left[ \left( 1 + \frac{r_m}{12} \right)^n - 1 \right] \quad . $  

## Calculating the months to retirement

Given the investment portfolio value $P_n$ after $n$ months of saving, we can evaluate when investment returns will sustain expenses using the safe withdrawal rate $r_w$.  This is typically understood as a yearly withdrawal rate, e.g. as stated in "the 4% rule," but for convenience I'm considering a safe monthly withdrawal rate.  I.e., if 4% is the yearly "safe withdrawal rate," then $r_w$ in this calculatuion would be something like (4 / 12)%, or 1/3% up to details related to monthly vs. yearly compounding.

## Function definitions

The only absolute scale in the problem is set by either the income, savings, or expenses rate.  As a matter of choice, I'm using the reference quantity as the monthly income.

In [2]:
def monthly_rate_from_yearly_rate(yearly_rate: float) -> float:
    """
    Convert a yearly compounding rate to a monthly compounding rate.
    - yearly_rate: the yearly compounding rate.  E.g., 8% yearly compounding has a yearly_rate=0.08.
    """
    return 12 * (math.pow(1 + yearly_rate, 1 / 12) - 1)

def compound_monthly(months_elapsed: int, monthly_rate: float) -> float:
    """
    Returns the proportional factor due to monthly compounding.
    - months_elapsed: the number of months to compound
    - monthly_rate: the rate for monthly compounding
    """
    return math.pow(1 + monthly_rate / 12, months_elapsed)

def investment_value(months_elapsed: int, savings_ratio: float, yearly_real_investment_rate: float) -> float:
    """
    Calculates the (real, inflation-adjusted) value of an investment portfolio, in units of the monthly real income, based on a constant savings rate.
    - months_elapsed: the number of months saved at the given rate
    - savings_ratio: the portion of income invested into the investment portfolio each month
    - yearly_real_investment_rate: the real rate of investment returns, when compounded yearly
    """
    monthly_rate = monthly_rate_from_yearly_rate(yearly_real_investment_rate)
    return savings_ratio * sum(
        [compound_monthly(month, monthly_rate) for month in range(1, months_elapsed + 1)]
        )

In [ ]:
# Benchmark comparison to https://www.mrmoneymustache.com/2012/01/13/the-shockingly-simple-math-behind-early-retirement/

years = 51
savings_ratio = 0.1
yearly_real_investment_return = 0.05
yearly_safe_withdrawal_rate = 0.04
monthly_safe_withdrawal_rate = yearly_safe_withdrawal_rate / 12  # Just an assumption

expense_ratio = 1 - savings_ratio
portfolio_value = investment_value(12 * years, savings_ratio, yearly_real_investment_return)
(portfolio_value * monthly_safe_withdrawal_rate) / expense_ratio

1.007783270662107

In [7]:
portfolio_value

272.1014830787689

In [23]:
monthly_rate = monthly_rate_from_yearly_rate(yearly_real_investment_return)
(savings_ratio * 12 / monthly_rate) * compound_monthly(1, monthly_rate) * (compound_monthly(12 * years, monthly_rate) - 1)

272.101483078769